# V3 Release Manifest

Builds the release manifest for the cache paper from the files in `E:\apriori-ml-v3`, and runs the counting checks of Section 4 on the CPU. No training, no measurement. The notebook is itself part of the release and regenerates `release\MANIFEST.md` on demand.

**Cell 1.** Walks the project once, keeps the release set defined by its `INCLUDE` patterns, computes the full SHA256 of every kept file, checks every library module against the hashes recorded in the pilot gate record before training, writes `MANIFEST.md` into `release\` and returns the listing as a DataFrame. Files the patterns do not match are printed so the release set can be corrected before the manifest is final.

**Cell 2.** Counting checks of Section 4.7, deterministic enumeration with no GPU and no data. It imports `set_attention_cache` for the unranking routine, the successor rule, the canonical trie builder and the run sampler, and nothing else from the library. It asserts the theorems on the enumerated cases and prints the tables that Section 4 cites. It writes no file, so its printed output is the record, compared line by line on a rerun.

**Release set.**

| Folder | Kept | Role |
|---|---|---|
| `src\` | every `.py`, including `apriori\serialization.py` | library and pilot source |
| `notebooks\` | `V3 Benchmarks 2080`, `V3 Benchmarks H100`, `V3 Pilot 2080`, `V3_Benchmarks_Tables`, this notebook | the notebooks that produced and tabulate every result |
| `inputs\` | every file | frozen benchmark inputs, 18 files |
| `results\` | `section7_benchmarks_2080.pkl.lz4`, `h100_v3\section7_benchmarks_H100.pkl.lz4`, `tables\` | benchmark results and the generated tables, paragraph and figure |
| `pilot\` | the four `mc3_*` data extracts, `runs\v3_*_curve.pkl.lz4` and `runs\v3_*_last.pt` for the three runs W0_registered, W0 and W, `v3_pilot_report`, `v3_cost_study`, `v3_gate`, `tables\` | pilot data, curves, final checkpoints, report, cost study, gate record and generated tables |
| root | `README.md` | r**Excluded.** Epoch checkpoints (`v3_*_epoch1.pt`, `v3_*_epoch2.pt`), `paper_subset_cache\`, `h100.zip`, `h100_stage\`, `README.txt`, `__pycache__`, `.ipynb_checkpoints` and the `release\` folder itself.se\` folder itself.

**Module hashes.** The pilot gate record `pilot\runs\v3_gate.pkl.lz4` holds the full SHA256 of every library module as hashed before training. Cell 1 compares each module on disk with that record and fails a mismatch visibly in the `pins` table rather than raising, so a stale file is seen, not hidden. It also prints the metadata keys of the two benchmark results files, so the module hashes they recorded can be compared with the same record.is seen, not hidden.

In [1]:
# Estimated run time: 1 to 3 minutes (sha256 over the inputs, pilot extracts, checkpoints and results; the inputs and extracts dominate)
import os, sys, hashlib, datetime, fnmatch
import pandas as pd

ROOT = r"E:\apriori-ml-v3"
SRC = os.path.join(ROOT, "src")
RELEASE_DIR = os.path.join(ROOT, "release")
MANIFEST = os.path.join(RELEASE_DIR, "MANIFEST.md")
GATE_RECORD = os.path.join(ROOT, "pilot", "runs", "v3_gate.pkl.lz4")
RESULT_FILES = ["results/section7_benchmarks_2080.pkl.lz4", "results/h100_v3/section7_benchmarks_H100.pkl.lz4"]

if SRC not in sys.path:
    sys.path.insert(0, SRC)
from apriori.serialization import load_from_pickle

# The H100 file was written by a newer pandas: StringDtype pickles with two constructor arguments and string arrays
# pickle their state as (dtype, data). This kernel's pandas takes one argument and (data, dtype, other). This loader
# accepts both forms. It is the loader of V3_Benchmarks_Tables.ipynb Cell 1.
import pickle, lz4.frame
import pandas as _pd
from pandas.core.arrays.string_ import StringArray as _StringArray

if not getattr(_pd.StringDtype, "_v2_shim", False):
    _orig_init = _pd.StringDtype.__init__
    def _shim_init(self, storage=None, *args, **kwargs):
        _orig_init(self, storage)
    _pd.StringDtype.__init__ = _shim_init
    _pd.StringDtype._v2_shim = True

class _StringArrayCompat(_StringArray):
    def __setstate__(self, state):
        if isinstance(state, tuple) and len(state) == 2 and isinstance(state[0], _pd.StringDtype):
            state = (state[1], state[0], {})
        super().__setstate__(state)

class _Unpickler(pickle.Unpickler):
    def find_class(self, module, name):
        if name.startswith("StringArray") and module.startswith("pandas"):
            return _StringArrayCompat
        if module.startswith("numpy._core"):
            module = "numpy.core" + module[len("numpy._core"):]
        return super().find_class(module, name)

def load_results(path):
    with lz4.frame.open(path, "rb") as f:
        return _Unpickler(f).load()

# release set: (glob relative to ROOT, release role). Order fixes the manifest order.
INCLUDE = [
    ("README.md",                                       "release README"),
    ("src/apriori/*.py",                                "serialization helper"),
    ("src/*.py",                                        "library and pilot source"),
    ("notebooks/V3 Benchmarks 2080.ipynb",              "benchmark notebook, 2080"),
    ("notebooks/V3 Benchmarks H100.ipynb",              "benchmark notebook, H100"),
    ("notebooks/V3 Pilot 2080.ipynb",                   "pilot notebook"),
    ("notebooks/V3_Benchmarks_Tables.ipynb",            "tables, paragraph and figure notebook"),
    ("notebooks/V3 Release Manifest.ipynb",             "release manifest notebook"),
    ("inputs/*",                                        "benchmark inputs"),
    ("results/section7_benchmarks_2080.pkl.lz4",        "benchmark results, 2080"),
    ("results/h100_v3/section7_benchmarks_H100.pkl.lz4", "benchmark results, H100"),
    ("results/tables/*",                                "generated benchmark tables, paragraph, figure and derived views"),
    ("pilot/mc3_pilot_data_d5.pkl.lz4",                 "pilot data extract"),
    ("pilot/mc3_contexts_d5.pkl.lz4",                   "pilot data extract"),
    ("pilot/mc3_decile_edges.pkl.lz4",                  "pilot data extract"),
    ("pilot/mc3_pilot_data_d5_removed.pkl.lz4",         "pilot data extract"),
    ("pilot/runs/v3_*_curve.pkl.lz4",                   "pilot curve"),
    ("pilot/runs/v3_*_last.pt",                         "pilot final checkpoint"),
    ("pilot/runs/v3_pilot_report.pkl.lz4",              "pilot report"),
    ("pilot/runs/v3_cost_study.pkl.lz4",                "pilot cost study"),
    ("pilot/runs/v3_gate.pkl.lz4",                      "pilot agreement gate record, module hashes before training"),
    ("pilot/tables/*",                                  "generated pilot tables"),
]
EXCLUDE_DIRS = {"__pycache__", "release", ".ipynb_checkpoints", "paper_subset_cache", "h100_stage", ".git"}
EXCLUDE_FILES = {"h100.zip", "README.txt"}

def excluded_dir(d):
    return d in EXCLUDE_DIRS or "superseded" in d.lower()

def sha256_file(path, block=1 << 22):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(block), b""):
            h.update(chunk)
    return h.hexdigest()

# ----- walk once, then match against the release set -----
all_files = []
for dirpath, dirnames, filenames in os.walk(ROOT):
    dirnames[:] = [d for d in dirnames if not excluded_dir(d)]
    for fn in filenames:
        if dirpath == ROOT and fn in EXCLUDE_FILES:
            continue
        rel = os.path.relpath(os.path.join(dirpath, fn), ROOT).replace("\\", "/")
        all_files.append(rel)

rows, seen = [], set()
for pattern, role in INCLUDE:
    hits = sorted(f for f in all_files if fnmatch.fnmatchcase(f, pattern) and f not in seen)
    if not hits:
        rows.append({"path": pattern, "role": role, "bytes": None, "sha256": "MISSING"})
        continue
    for f in hits:
        seen.add(f)
        p = os.path.join(ROOT, f)
        rows.append({"path": f, "role": role, "bytes": os.path.getsize(p), "sha256": sha256_file(p)})
manifest = pd.DataFrame(rows)
manifest["sha16"] = manifest["sha256"].str[:16]
manifest["modified"] = [datetime.datetime.fromtimestamp(os.path.getmtime(os.path.join(ROOT, r.path))).strftime("%Y-%m-%d %H:%M")
                        if r.sha256 != "MISSING" else None for r in manifest.itertuples()]

# ----- module hashes against the gate record -----
gate = load_from_pickle(GATE_RECORD)
PINS = {f"src/{f}": h for f, h in gate["sources_sha256"].items()}    # full SHA256 per library module, hashed before training
pin_rows = []
for f, want in PINS.items():
    got = manifest.loc[manifest["path"] == f, "sha256"]
    pin_rows.append({"file": f, "recorded_sha16": want[:16], "disk_sha16": got.iloc[0][:16] if len(got) else "MISSING",
                     "ok": bool(len(got)) and got.iloc[0] == want})
pins = pd.DataFrame(pin_rows)

# ----- benchmark results metadata, for the hash comparison of the benchmark side -----
for rel in RESULT_FILES:
    store = load_results(os.path.join(ROOT, rel))
    top = list(store.keys()) if isinstance(store, dict) else type(store).__name__
    meta = store.get("metadata") if isinstance(store, dict) else None
    print(f"\n{rel}\n  top level keys: {top}")
    if isinstance(meta, dict):
        print(f"  metadata keys: {list(meta.keys())}")
        for k, v in meta.items():
            if isinstance(v, dict) and any(isinstance(x, str) and len(x) == 64 for x in v.values()):
                print(f"  {k}:")
                for kk, vv in v.items():
                    print(f"    {kk:40s} {str(vv)[:16]}")

unmatched = sorted(f for f in all_files if f not in seen)

print("\nrelease files:", int((manifest["sha256"] != "MISSING").sum()), " missing patterns:", int((manifest["sha256"] == "MISSING").sum()))
print(manifest[["path", "role", "bytes", "sha16", "modified"]].to_string(index=False))
print("\npins, disk against the gate record"); print(pins.to_string(index=False))
print("\nproject files not in the release set:", len(unmatched))
for f in unmatched: print("  ", f)

# ----- write MANIFEST.md -----
os.makedirs(RELEASE_DIR, exist_ok=True)
lines = ["# Release manifest", "",
         f"Generated {datetime.datetime.now():%Y-%m-%d %H:%M}. Paths are relative to the release root. SHA256 is the full digest.", "",
         "## Module hashes", "", "Full SHA256 of each library module as recorded in the pilot gate record before training, and as found on disk.", "",
         "| File | Recorded SHA256 | On disk | Match |", "|---|---|---|---|"]
for r in pins.itertuples():
    disk = manifest.loc[manifest["path"] == r.file, "sha256"]
    lines.append(f"| {r.file} | {PINS[r.file]} | {disk.iloc[0] if len(disk) else 'MISSING'} | {'yes' if r.ok else 'NO'} |")
lines += ["", "## Files", "", "| Path | Role | Bytes | SHA256 |", "|---|---|---|---|"]
for r in manifest.itertuples():
    lines.append(f"| {r.path} | {r.role} | {r.bytes if r.bytes is not None else ''} | {r.sha256} |")
with open(MANIFEST, "w", encoding="utf-8") as f:
    f.write("\n".join(lines) + "\n")
print("\nwrote", MANIFEST)
manifest


results/section7_benchmarks_2080.pkl.lz4
  top level keys: ['benchmarks', 'method_evaluations', 'execution_calls', 'agreement_checks', 'preparation_measurements', 'metadata']
  metadata keys: ['schema_version', 'machine', 'manifest', 'manifest_sha256', 'implementation_commit', 'implementation_file_sha256', 'machine_environment', 'tolerances', 'timing_boundaries', 'agreement_rule', 'replay_verification_rule', 'budget_rule', 'ceiling_ms', 'suite_budget_h', 'warmups', 'timed_repeats', 'preparation_repeats', 'graph_warmups', 'suite_elapsed_s', 'created', 'mock']
  implementation_file_sha256:
    set_attention.py                         8580eb3ce583edec
    set_attention_cache.py                   86d8f308bbbc1a79
    set_attention_data.py                    fd4c5a486477ccc3
    set_attention_model.py                   d7a7e72a2a111bb4
    set_attention_bench.py                   80b9a7eddb88ddee

results/h100_v3/section7_benchmarks_H100.pkl.lz4
  top level keys: ['benchmarks', 'method_eva

,path,role,bytes,sha256,sha16,modified
0,README.md,release README,19542,d0ebbafc335458a40700f6bf2725c6259014d7b8929f18...,d0ebbafc335458a4,2026-11-10 12:56
1,src/apriori/__init__.py,serialization helper,0,e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b93...,e3b0c44298fc1c14,2026-10-19 15:01
2,src/apriori/serialization.py,serialization helper,5429,d3f1036aef823647c63dcb985bcbb99217e3c008bff9d7...,d3f1036aef823647,2026-08-14 13:43
3,src/masked_concept_model.py,library and pilot source,22834,f6b49be2e07791e1e8f39729f881e4488a85943f8de35a...,f6b49be2e07791e1,2026-10-15 14:18
4,src/mc_batched.py,library and pilot source,9395,571185cc24b467ee1955561b19348522548e328ecd7b0a...,571185cc24b467ee,2026-10-30 11:00
...,...,...,...,...,...,...
61,pilot/runs/v3_cost_study.pkl.lz4,pilot cost study,3740,dab57d5e88a2f098de28f765ddfe88b37a53c0a0e9b279...,dab57d5e88a2f098,2026-11-10 10:24
62,pilot/runs/v3_gate.pkl.lz4,"pilot agreement gate record, module hashes bef...",3869,e2da89f20064f75dab24f0bcff57e717e8a60a15a020ca...,e2da89f20064f75d,2026-11-09 14:15
63,pilot/tables/tab_pilot_baseline.tex,generated pilot tables,1022,94c31e875669c53d77f3c9fe1d16c28356b0a8933fa6a6...,94c31e875669c53d,2026-11-10 10:24
64,pilot/tables/tab_pilot_bins_sampled.tex,generated pilot tables,1760,c3112c18b7c6f16d79c93989b305550732013eaef48e22...,c3112c18b7c6f16d,2026-11-10 10:24


## Cell 2. Counting checks of Section 4

Deterministic CPU enumeration of the statements in Section 4.7. Every check either asserts or prints a table. Printed labels:

- `order`: the recursive preorder listing of the trie of all nonempty subsets agrees with `unrank_lex_subset` at every rank, n = 2 to 12, and the successor rule from the first subset reproduces it, n = 1 to 10, with no successor after the last subset.
- `theorems`: node counts of the canonical trie equal `binom(n+1, k) - 1` for every k and `2^n - 1` for the full power set, n = 2 to 12.
- `nonwrapping runs`: every run of every length at n = 3 to 10 has exactly `P + |f| - 1` nodes. Prints the number of runs checked per n, violations and slack.
- `run examples`: first and last runs of lengths 1, 2, 5, 17, 100 and 1,000 at n = 10, 16 and 20, wrapping runs of lengths 2, 5, 17, 100 and 1,000, and the last run of 1,000 at n = 100, each with nodes `N`, feature contributions `F`, ratio `N / F` and the bound `P + |f| - 1`.
- `four run samples`: R = 4 runs of length L at n = 4 to 10 and L in {2, 4, 8, 16} with 4L <= T, over every offset. `mean_F` against `L / exr(n)`, `ratio_of_means` against the bound `exr(n) (1 + (n - 1) / L)`, and the mean and maximum realized ratio.
- `run sampler cases`: n = 2 to 8, budgets 1, 2, 3, floor(T/3), floor(T/2) and T - 1 with 1 <= M < T (distinct values), run lengths 1, 2, 3, 5, 8 and 64, every offset. Each draw has P = RL distinct ranks and every rank is included for exactly P of the T offsets. Prints the case count.
- `seeded sampler`: `sample_subsets_powerset_runs` returns sorted distinct subsets of length P equal to the runs under one offset, and every subset when M >= T.

Estimated run time: 1 to 3 minutes.

In [2]:
# Estimated run time: 1 to 3 minutes (the nonwrapping run check at n = 10 enumerates 523,776 runs; the rest is seconds).
# Counting checks of Section 4.7. CPU only, no data, no file written. Fresh kernel or after cell 1, either works.
import os, sys
from math import comb
import numpy as np
import pandas as pd

ROOT = r"E:\apriori-ml-v3"
SRC = os.path.join(ROOT, "src")
if SRC not in sys.path:
    sys.path.insert(0, SRC)
from set_attention_cache import (unrank_lex_subset, canonical_successor, build_prefix_trie_canonical,
                                 powerset_run_base, powerset_run_ranks, sample_subsets_powerset_runs)

def T_of(n):
    return (1 << n) - 1

def exr(n):
    return T_of(n) / (n * 2 ** (n - 1))

def preorder(n):
    """Recursive preorder of the trie of all nonempty subsets of range(n): each prefix before its extensions."""
    out = []
    def visit(prefix, start):
        for e in range(start, n):
            S = prefix + (e,)
            out.append(S)
            visit(S, e + 1)
    visit((), 0)
    return out

def nodes_of(subsets):
    """Number of distinct nonempty prefixes, the trie node count, for subsets in any order."""
    seen = set()
    for S in subsets:
        for j in range(1, len(S) + 1):
            seen.add(S[:j])
    return len(seen)

def facts_of(subsets):
    return sum(len(S) for S in subsets)

def run_subsets(n, start, L):
    """L consecutive ranks from start, modulo T, as subsets."""
    T = T_of(n)
    return [unrank_lex_subset((start + i) % T, n) for i in range(L)]

# ----- order: preorder against unranking, successor rule -----
for n in range(2, 13):
    T = T_of(n)
    po = preorder(n)
    assert len(po) == T
    assert all(po[r] == unrank_lex_subset(r, n) for r in range(T)), f"preorder differs from unranking at n = {n}"
for n in range(1, 11):
    T = T_of(n)
    S = unrank_lex_subset(0, n)
    for r in range(T):
        assert S == unrank_lex_subset(r, n), f"successor rule differs from unranking at n = {n}, rank {r}"
        S = canonical_successor(S, n)
    assert S is None, f"last subset has a successor at n = {n}"
print("order: preorder equals unranking for n = 2 to 12; successor rule reproduces unranking for n = 1 to 10, no successor after the last subset")

# ----- theorems: subsets of one size, full power set -----
rows = []
for n in range(2, 13):
    T = T_of(n)
    full = [unrank_lex_subset(r, n) for r in range(T)]
    N_full = len(build_prefix_trie_canonical(full)["pos"])
    assert N_full == T, f"power set trie has {N_full} nodes at n = {n}, expected {T}"
    for k in range(1, n + 1):
        sub = [S for S in full if len(S) == k]
        N_k = len(build_prefix_trie_canonical(sub)["pos"])
        want = comb(n + 1, k) - 1
        assert N_k == want, f"size {k} trie has {N_k} nodes at n = {n}, expected {want}"
    rows.append({"n": n, "T": T, "power_set_nodes": N_full, "sizes_checked": n})
theorems = pd.DataFrame(rows)
print("\ntheorems: every k at n = 2 to 12 and the full power set match, see table")
print(theorems.to_string(index=False))

# ----- nonwrapping runs: every run of every length, n = 3 to 10 -----
rows = []
for n in range(3, 11):
    T = T_of(n)
    full = [unrank_lex_subset(r, n) for r in range(T)]
    runs = violations = slack = 0
    for a in range(T):
        seen = set()
        f_len = len(full[a])
        for b in range(a, T):
            S = full[b]
            for j in range(1, len(S) + 1):
                seen.add(S[:j])
            P = b - a + 1
            want = P + f_len - 1
            runs += 1
            if len(seen) != want:
                violations += 1
                slack += want - len(seen)
    rows.append({"n": n, "runs": runs, "violations": violations, "slack": slack})
nonwrap = pd.DataFrame(rows)
assert (nonwrap.violations == 0).all() and (nonwrap.slack == 0).all()
print("\nnonwrapping runs: Theorem runs holds with equality for every run")
print(nonwrap.to_string(index=False))

# ----- run examples: first, last and wrapping runs -----
rows = []
LENS = [1, 2, 5, 17, 100, 1000]
for n in (10, 16, 20):
    T = T_of(n)
    for L in LENS:
        if L > T:
            continue
        for kind, start in (("first", 0), ("last", T - L)):
            subs = run_subsets(n, start, L)
            N, F = nodes_of(subs), facts_of(subs)
            rows.append({"n": n, "kind": kind, "L": L, "N": N, "F": F, "ratio": N / F, "bound": L + len(subs[0]) - 1,
                         "exr": exr(n), "ratio_over_exr": (N / F) / exr(n)})
        if L >= 2:
            h = L // 2
            subs = run_subsets(n, T - h, L)                      # last h ranks then the first L - h, wrapping at T
            N, F = nodes_of(subs), facts_of(subs)
            rows.append({"n": n, "kind": "wrapping", "L": L, "N": N, "F": F, "ratio": N / F, "bound": L + len(subs[0]) - 1,
                         "exr": exr(n), "ratio_over_exr": (N / F) / exr(n)})
n = 100
subs = run_subsets(n, T_of(n) - 1000, 1000)
N, F = nodes_of(subs), facts_of(subs)
rows.append({"n": n, "kind": "last", "L": 1000, "N": N, "F": F, "ratio": N / F, "bound": 1000 + len(subs[0]) - 1,
             "exr": exr(n), "ratio_over_exr": (N / F) / exr(n)})
examples = pd.DataFrame(rows)
nonwrapping_rows = examples[examples.kind != "wrapping"]
assert (nonwrapping_rows.N == nonwrapping_rows.bound).all(), "a first or last run misses Theorem runs"
wrapping_rows = examples[examples.kind == "wrapping"]
assert (wrapping_rows.N <= wrapping_rows.bound).all(), "a wrapping run exceeds the bound"
print("\nrun examples: first and last runs meet Theorem runs with equality, wrapping runs satisfy the bound")
with pd.option_context("display.float_format", "{:.4f}".format):
    print(examples.to_string(index=False))

# ----- four run samples over every offset -----
rows = []
R = 4
for n in range(4, 11):
    T = T_of(n)
    full = [unrank_lex_subset(r, n) for r in range(T)]
    gap = T // R
    for L in (2, 4, 8, 16):
        if R * L > T:
            continue
        Ns, Fs = [], []
        for o in range(T):
            subs = [full[(o + j * gap + i) % T] for j in range(R) for i in range(L)]
            Ns.append(nodes_of(subs))
            Fs.append(facts_of(subs))
        Ns, Fs = np.asarray(Ns, dtype=float), np.asarray(Fs, dtype=float)
        mean_F_run = Fs.mean() / R
        bound = exr(n) * (1 + (n - 1) / L)
        realized = Ns / Fs
        rows.append({"n": n, "L": L, "offsets": T, "mean_F_per_run": mean_F_run, "L_over_exr": L / exr(n),
                     "ratio_of_means": Ns.mean() / Fs.mean(), "bound": bound, "mean_realized": realized.mean(),
                     "max_realized": realized.max()})
samples = pd.DataFrame(rows)
assert np.allclose(samples.mean_F_per_run, samples.L_over_exr, atol=1e-9, rtol=0), "mean feature contributions differ from L / exr(n)"
assert (samples.ratio_of_means <= samples.bound).all(), "ratio of means exceeds the bound"
exceed = samples[samples.max_realized > samples.bound]
print(f"\nfour run samples: mean F per run equals L / exr(n) within 1e-9, ratio of means below the bound in every row, "
      f"max realized ratio exceeds the bound in {len(exceed)} rows")
with pd.option_context("display.float_format", "{:.6f}".format):
    print(samples.to_string(index=False))

# ----- run sampler: P = RL distinct ranks, inclusion P of T offsets -----
cases = 0
for n in range(2, 9):
    T = T_of(n)
    budgets = sorted({M for M in (1, 2, 3, T // 3, T // 2, T - 1) if 1 <= M < T})
    for M in budgets:
        for lam in (1, 2, 3, 5, 8, 64):
            T_, Rn, L, s = powerset_run_base(n, M, lam)
            P = Rn * L
            hits = np.zeros(T, dtype=np.int64)
            for o in range(T):
                ranks = [(start + i) % T for start, Ln in powerset_run_ranks(n, M, lam, o) for i in range(Ln)]
                assert len(ranks) == P and len(set(ranks)) == P, f"n {n} M {M} lambda {lam} offset {o}: ranks not P distinct"
                hits[ranks] += 1
            assert (hits == P).all(), f"n {n} M {M} lambda {lam}: a rank is not included for exactly P offsets"
            cases += 1
print(f"\nrun sampler cases: {cases} cases over n = 2 to 8, every offset, P = RL distinct ranks and inclusion exactly P of T")

# ----- seeded sampler output -----
checked = 0
for n in range(3, 9):
    T = T_of(n)
    full = [unrank_lex_subset(r, n) for r in range(T)]
    rank_of = {S: r for r, S in enumerate(full)}
    for M in sorted({2, 3, T // 2, T - 1}):
        for lam in (1, 3, 8):
            out = sample_subsets_powerset_runs(n, f"count_check_n{n}", M, lam)
            T_, Rn, L, s = powerset_run_base(n, M, lam)
            P = Rn * L
            assert out == sorted(out) and len(set(out)) == len(out) == P, f"n {n} M {M} lambda {lam}: output not sorted distinct of length P"
            got = set(rank_of[S] for S in out)
            match = [o for o in range(T) if set((start + i) % T for start, Ln in powerset_run_ranks(n, M, lam, o) for i in range(Ln)) == got]
            assert match, f"n {n} M {M} lambda {lam}: output is not the runs under any offset"
            checked += 1
    assert sample_subsets_powerset_runs(n, "all", T, 8) == full, f"M >= T does not return every subset at n = {n}"
print(f"seeded sampler: {checked} draws equal the runs under one offset, sorted and distinct; M >= T returns every subset")
print("\nCOUNTING CHECKS PASSED")
examples

order: preorder equals unranking for n = 2 to 12; successor rule reproduces unranking for n = 1 to 10, no successor after the last subset

theorems: every k at n = 2 to 12 and the full power set match, see table
 n    T  power_set_nodes  sizes_checked
 2    3                3              2
 3    7                7              3
 4   15               15              4
 5   31               31              5
 6   63               63              6
 7  127              127              7
 8  255              255              8
 9  511              511              9
10 1023             1023             10
11 2047             2047             11
12 4095             4095             12

nonwrapping runs: Theorem runs holds with equality for every run
 n   runs  violations  slack
 3     28           0      0
 4    120           0      0
 5    496           0      0
 6   2016           0      0
 7   8128           0      0
 8  32640           0      0
 9 130816           0      0
10 523776 

,n,kind,L,N,F,ratio,bound,exr,ratio_over_exr
0,10,first,1,1,1,1.000000,1,0.199805,5.004888
1,10,last,1,1,1,1.000000,1,0.199805,5.004888
2,10,first,2,2,3,0.666667,2,0.199805,3.336592
3,10,last,2,3,3,1.000000,3,0.199805,5.004888
4,10,wrapping,2,2,2,1.000000,2,0.199805,5.004888
5,10,first,5,5,15,0.333333,5,0.199805,1.668296
6,10,last,5,7,9,0.777778,7,0.199805,3.892690
7,10,wrapping,5,6,9,0.666667,6,0.199805,3.336592
8,10,first,17,17,113,0.150442,17,0.199805,0.752948
9,10,last,17,19,37,0.513514,19,0.199805,2.570077
